# Building an Autogen Agent with Zep Long-term Memory

This notebook restores the educational Autogen + Zep flow using the **current thread/user/graph APIs**:
- Preload prior conversation into a Zep thread
- Bounded wait for ingestion
- A `ConversableAgent` that reads/writes Zep memory each turn
- Demonstrate `thread.get_user_context` and `graph.search`

Requires `ZEP_API_KEY` and `OPENAI_API_KEY`.

## Install

Pin classic Autogen (`ConversableAgent`) which remains the supported API for this example:

```bash
pip install "pyautogen>=0.2.35,<0.3" "zep-cloud>=3.28,<4" python-dotenv
```

In [ ]:
# %pip install -q "pyautogen>=0.2.35,<0.3" "zep-cloud>=3.28,<4" python-dotenv

In [ ]:
import asyncio
import os
import time
import uuid

from autogen import ConversableAgent
from dotenv import load_dotenv
from zep_cloud.client import AsyncZep
from zep_cloud.types import Message

load_dotenv()
assert os.environ.get("ZEP_API_KEY"), "ZEP_API_KEY is required"
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is required"

zep = AsyncZep(api_key=os.environ["ZEP_API_KEY"])
llm_config = {
    "config_list": [
        {"model": "gpt-4o-mini", "api_key": os.environ["OPENAI_API_KEY"]}
    ]
}

## Bounded ingestion wait

Poll the task returned by `thread.add_messages` until Zep finishes processing.

In [ ]:
async def wait_for_task(
    task_id: str | None,
    *,
    timeout_seconds: float = 60.0,
    poll_interval_seconds: float = 2.0,
) -> None:
    if not task_id:
        return
    deadline = time.monotonic() + timeout_seconds
    while True:
        task = await zep.task.get(task_id)
        status = (getattr(task, "status", None) or "").lower()
        if status in {"succeeded", "completed", "complete", "success"}:
            print(f"Task {task_id} {status}")
            return
        if status in {"failed", "error", "canceled", "cancelled", "partial"}:
            raise RuntimeError(f"Zep task {task_id} failed: {getattr(task, 'error', None)}")
        if time.monotonic() >= deadline:
            raise TimeoutError(
                f"Timed out waiting for task {task_id} after {timeout_seconds}s"
            )
        await asyncio.sleep(poll_interval_seconds)


## ZepConversableAgent

Each turn: persist the user message, refresh context from Zep, then persist the assistant reply.

In [ ]:
class ZepConversableAgent(ConversableAgent):
    """ConversableAgent with Zep thread memory on every turn."""

    def __init__(self, name, zep_client, zep_thread_id, zep_user_name, **kwargs):
        super().__init__(name=name, **kwargs)
        self.zep_client = zep_client
        self.zep_thread_id = zep_thread_id
        self.zep_user_name = zep_user_name
        self._base_system = kwargs.get("system_message", "")

    async def a_generate_reply(self, messages=None, sender=None, **kwargs):
        messages = messages or self.chat_messages[sender]
        last = messages[-1]
        added = await self.zep_client.thread.add_messages(
            thread_id=self.zep_thread_id,
            messages=[
                Message(
                    role="user",
                    name=self.zep_user_name,
                    content=last.get("content", ""),
                )
            ],
        )
        await wait_for_task(getattr(added, "task_id", None), timeout_seconds=30.0)

        memory = await self.zep_client.thread.get_user_context(
            thread_id=self.zep_thread_id
        )
        context = memory.context or ""
        self.update_system_message(
            self._base_system
            + f"\n\nRelevant long-term memory from Zep:\n{context}"
        )

        reply = await super().a_generate_reply(messages=messages, sender=sender, **kwargs)
        if reply:
            added = await self.zep_client.thread.add_messages(
                thread_id=self.zep_thread_id,
                messages=[
                    Message(role="assistant", name=self.name, content=str(reply))
                ],
            )
            await wait_for_task(getattr(added, "task_id", None), timeout_seconds=30.0)
        return reply

## Create user/thread and preload prior conversation

In [ ]:
user_name = "Cathy"
user_id = user_name + uuid.uuid4().hex[:4]
thread_id = str(uuid.uuid4())

await zep.user.add(
    user_id=user_id, first_name=user_name, email=f"{user_id}@example.com"
)
await zep.thread.create(thread_id=thread_id, user_id=user_id)

prior = [
    Message(
        role="assistant",
        name="CareBot",
        content=f"Hi {user_name}, how are you feeling today?",
    ),
    Message(
        role="user",
        name=user_name,
        content="I've been grieving my mother. Some days are heavy.",
    ),
    Message(
        role="assistant",
        name="CareBot",
        content="I'm sorry for your loss. What usually helps on hard days?",
    ),
    Message(
        role="user",
        name=user_name,
        content="Short walks and looking through old photos.",
    ),
]
added = await zep.thread.add_messages(thread_id=thread_id, messages=prior)
await wait_for_task(getattr(added, "task_id", None), timeout_seconds=60.0)
print({"user_id": user_id, "thread_id": thread_id})

## Run a short non-interactive conversation, then inspect memory + search

In [ ]:
carebot = ZepConversableAgent(
    name="CareBot",
    zep_client=zep,
    zep_thread_id=thread_id,
    zep_user_name=user_name,
    system_message=(
        "You are a compassionate caregiver. Use Zep memory about the user when helpful."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)
cathy = ConversableAgent(
    name=user_name,
    system_message="You are Cathy. Keep replies brief and in character.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

chat_result = await carebot.a_initiate_chat(
    cathy,
    message="Hi Cathy, nice to see you again. How are you holding up?",
    max_turns=2,
)
print(chat_result)

context = await zep.thread.get_user_context(thread_id=thread_id)
print("--- get_user_context ---")
print(context.context)

search = await zep.graph.search(
    user_id=user_id, query="family", limit=3, scope="edges"
)
print("--- graph.search edges ---")
print(getattr(search, "edges", search))